In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PATCH_DIR = ROOT / "datasets/vnp46a2/poi_patches_5x5"
GHSL_CSV = (
    ROOT / "datasets/vnp46a2/settlement_daily_stats"
    / "SamarLeyte_DNBBRDF_AllClasses_DailyStats_2012_2024.csv"
)
FIG_DIR = ROOT / "outputs/figures/supplementary/source_signal_2013"
TABLE_DIR = ROOT / "outputs/tables/supplementary/source_signal_2013"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

DATES = pd.date_range("2013-01-01", "2013-12-31", freq="D")
HAIYAN = pd.Timestamp("2013-11-08").to_pydatetime()

POIS = {
    "Tacloban_City": "Tacloban City Center",
    "Ormoc_City": "Ormoc City Center",
    "Baybay_City": "Baybay City Center",
    "Palo": "Palo",
    "Alang_Alang": "Alang Alang",
    "Guiuan": "Guiuan",
    "Samar_Forest_Reserve": "Samar Forest Reserve",
    "Mt_Nacolod_Upland": "Mt. Nacolod Upland",
    "Central_Leyte_Forest": "Central Leyte Forest",
    "Southern_Leyte_Gulf": "Southern Leyte Gulf",
}

GHSL_CLASSES = {
    "10": "Water",
    "11": "Very low-density rural",
    "12": "Low-density rural",
    "13": "Rural cluster",
    "21": "Suburban / peri-urban",
    "22": "Semi-dense urban cluster",
    "23": "Dense urban cluster",
    "30": "Urban centre",
}

KERNELS = (1, 3, 5)
KERNEL_LABELS = {1: "1 × 1", 3: "3 × 3", 5: "5 × 5"}

In [ ]:
POI_CSV = (
    ROOT / "outputs/tables/supplementary"
    / "table_sx_poi_patch_daily_metrics_long.csv"
)

if not POI_CSV.is_file():
    raise FileNotFoundError(
        f"{POI_CSV}\nRun Notebook 01 through its POI long-table export cell."
    )

poi = pd.read_csv(POI_CSV)

required = {
    "date", "location", "extent",
    "DNB_mean", "Gap_mean", "DNB_valid_px",
}
missing = required - set(poi.columns)
if missing:
    raise ValueError(f"{POI_CSV.name}: missing {sorted(missing)}")

poi["date"] = pd.to_datetime(poi["date"], errors="raise")
poi = poi[poi["date"].isin(DATES)].copy()
poi = poi[poi["location"].isin(POIS)].copy()
poi = poi[poi["extent"].isin({"1x1", "3x3", "5x5"})].copy()

if poi.duplicated(["location", "extent", "date"]).any():
    raise ValueError("Duplicate POI × extent × date rows")

poi["direct"] = pd.to_numeric(poi["DNB_mean"], errors="coerce")
poi["gap"] = pd.to_numeric(poi["Gap_mean"], errors="coerce")
poi["valid_direct_px"] = pd.to_numeric(
    poi["DNB_valid_px"], errors="coerce"
).fillna(0)

kernel_pixels = {"1x1": 1, "3x3": 9, "5x5": 25}
poi["SC"] = (
    100 * poi["valid_direct_px"] / poi["extent"].map(kernel_pixels)
)
poi["direct"] = poi["direct"].where(poi["valid_direct_px"] > 0)

POI_DAILY = (
    poi.rename(columns={"extent": "kernel"})
    .set_index(["location", "kernel", "date"])
    .sort_index()
)
POI_DAILY.to_csv(TABLE_DIR / "poi_daily_source_signals_2013.csv")

def poi_panel(location, size):
    kernel = f"{size}x{size}"
    return (
        POI_DAILY.xs((location, kernel), level=("location", "kernel"))
        .reindex(DATES)
        .rename_axis("date")
    )

print(
    f"Loaded {POI_DAILY.index.get_level_values('location').nunique()} POIs "
    f"from Notebook 01's extracted table."
)

In [ ]:
if not GHSL_CSV.is_file():
    raise FileNotFoundError(GHSL_CSV)

ghsl = pd.read_csv(GHSL_CSV)

required = {"date", "ClassCode", "Mean_by_sum", "Valid_px"}
missing = required - set(ghsl.columns)
if missing:
    raise ValueError(f"{GHSL_CSV.name}: missing {sorted(missing)}")

ghsl["date"] = pd.to_datetime(ghsl["date"], errors="raise")
ghsl["ClassCode"] = (
    pd.to_numeric(ghsl["ClassCode"], errors="raise")
    .astype("Int64")
    .astype(str)
)
ghsl = ghsl[ghsl["date"].isin(DATES)].copy()
ghsl = ghsl[ghsl["ClassCode"].isin(GHSL_CLASSES)].copy()

if ghsl.duplicated(["ClassCode", "date"]).any():
    raise ValueError("Duplicate GHSL class × date records")

ghsl["Valid_px"] = pd.to_numeric(ghsl["Valid_px"], errors="coerce")
ghsl["direct"] = pd.to_numeric(ghsl["Mean_by_sum"], errors="coerce")
ghsl["direct"] = ghsl["direct"].where(ghsl["Valid_px"] > 0)

# Notebook 02's visual convention: relative class-level support.
ghsl["SC_denominator"] = (
    ghsl.groupby("ClassCode")["Valid_px"].transform("max")
)
ghsl["SC"] = np.where(
    ghsl["SC_denominator"] > 0,
    100 * ghsl["Valid_px"] / ghsl["SC_denominator"],
    np.nan,
)

# The documented class CSV has no class-level Gap-Filled field.
ghsl["gap"] = np.nan

GHSL_DAILY = ghsl.set_index(["ClassCode", "date"]).sort_index()
GHSL_DAILY.to_csv(TABLE_DIR / "ghsl_class_direct_signals_2013.csv")

def ghsl_panel(code):
    return (
        GHSL_DAILY.xs(code, level="ClassCode")
        .reindex(DATES)
        .rename_axis("date")
    )

found_classes = set(GHSL_DAILY.index.get_level_values("ClassCode"))
print(f"Loaded {len(found_classes)} GHSL classes for 2013.")

In [ ]:
def plot_source_signal(title, panels, filename, direct_color="#064CEB", show_median_sd=False):
    """
    panels = [(short_panel_title, dataframe), ...]
    Each dataframe has direct, gap, SC and a daily DatetimeIndex.

    A qualifying day has direct radiance and SC >= 50%.
    TC is the percentage of DATES that qualify.
    Max dropout is the longest consecutive run that does not qualify.
    """
    tau = 50.0
    ncols = len(panels)
    width = 1450 if ncols == 3 else 1000
    height = 300

    prepared = []
    for name, panel in panels:
        d = panel.reindex(DATES)
        direct = pd.to_numeric(d["direct"], errors="coerce")
        gap = pd.to_numeric(d["gap"], errors="coerce")
        sc = pd.to_numeric(d["SC"], errors="coerce")

        qualifies = direct.notna() & sc.ge(tau)
        tc = 100 * qualifies.mean()

        # Find consecutive days without a qualifying direct observation.
        dropout = (~qualifies).to_numpy()
        changes = np.diff(np.r_[False, dropout, False].astype(int))
        starts = np.flatnonzero(changes == 1)
        ends = np.flatnonzero(changes == -1)  # exclusive

        if len(starts):
            longest = int(np.argmax(ends - starts))
            max_days = int(ends[longest] - starts[longest])
            max_start = DATES[starts[longest]]
            max_end = DATES[ends[longest] - 1] + pd.Timedelta(days=1)
        else:
            max_days = 0
            max_start = max_end = None

        prepared.append(
            (name, direct, gap, sc, tc, max_days, max_start, max_end)
        )

    fig = make_subplots(
        rows=2,
        cols=ncols,
        shared_xaxes="columns",
        shared_yaxes=(ncols > 1),
        row_heights=[0.16, 0.84],
        vertical_spacing=0.06,
        horizontal_spacing=0.055 if ncols > 1 else 0.02,
        subplot_titles=[item[0] for item in prepared] + [""] * ncols,
    )

    radiance_values = []

    for col, (_, direct, gap, sc, _, max_days, max_start, max_end) in enumerate(
        prepared, start=1
    ):
        radiance_values.extend(direct.dropna().tolist())
        radiance_values.extend(gap.dropna().tolist())

        fig.add_trace(
            go.Heatmap(
                x=DATES,
                y=[0],
                z=[sc.fillna(0).to_numpy(dtype=float)],
                coloraxis="coloraxis",
                xgap=0,
                ygap=0,
                hovertemplate="%{x|%d %b %Y}<br>SC %{z:.0f}%<extra></extra>",
            ),
            row=1, col=col,
        )

        # Shade only the longest below-threshold run, in the SC strip.
        if max_days:
            fig.add_vrect(
                x0=max_start,
                x1=max_end,
                fillcolor="rgba(238, 67, 180, 0.42)",
                line_color="#D32F9D",
                line_width=1,
                layer="above",
                row=1, col=col,
            )

            fig.add_annotation(
                x=max_start + (max_end - max_start) / 2,
                y=-0.5,
                xref="x" if col == 1 else f"x{col}",
                yref="y" if col == 1 else f"y{col}",
                text=f"{max_days} d",
                showarrow=False,
                yshift=-6,
                yanchor="top",
                font=dict(color="#C42C91", size=10),
            )

        if show_median_sd and direct.notna().sum() >= 2:
            median_ntl = float(direct.median())
            sd_ntl = float(direct.std(ddof=1))

            fig.add_trace(
                go.Scatter(
                    x=[DATES[0], DATES[-1]],
                    y=[median_ntl, median_ntl],
                    mode="lines",
                    showlegend=False,
                    hoverinfo="skip",
                    line=dict(color="black", width=1.5, dash="dot"),
                ),
                row=2, col=col,
            )

            radiance_axis = ncols + col
            fig.add_annotation(
                x=1.005,                  # same paper x-position as SC colorbar
                y=median_ntl,
                xref="paper",
                yref=f"y{radiance_axis}",
                text=f"{median_ntl:.1f} ± {sd_ntl:.1f}",
                showarrow=False,
                xanchor="left",
                yanchor="middle",
                font=dict(color="black", size=10),
            )
            
        if gap.notna().any():
            fig.add_trace(
                go.Scatter(
                    x=DATES,
                    y=gap,
                    mode="lines",
                    name="Gap-Filled",
                    showlegend=(col == 1),
                    line=dict(color="#D94B40", width=1.1, shape="hv"),
                    connectgaps=False,
                ),
                row=2, col=col,
            )

        fig.add_trace(
            go.Scatter(
                x=DATES,
                y=direct,
                mode="markers",
                name="DNB-BRDF",
                showlegend=(col == 1),
                marker=dict(
                    color=direct_color,              # GHSL pixel color
                    size=6,
                    opacity=1,
                    line=dict(color="black", width=0.8),
                ),
            ),
            row=2, col=col,
        )

        # TC label: upper left of this radiance panel.
        radiance_axis = ncols + col
        fig.add_annotation(
            x=0.02,
            y=1.01,
            xref=f"x{radiance_axis} domain",
            yref=f"y{radiance_axis} domain",
            text=f"TC₅₀ = {prepared[col - 1][4]:.1f}%",
            showarrow=False,
            xanchor="left",
            yanchor="top",
            font=dict(color="#233B60", size=12),
            bgcolor="rgba(255,255,255,0.82)",
            borderpad=2,
        )
        
        for row in (1, 2):
            fig.add_vline(
                x=HAIYAN,
                row=row, col=col,
                line_color="#233B60",
                line_dash="dash",
                line_width=1.3,
            )

        fig.update_yaxes(
            range=[-0.5, 0.5],
            showticklabels=False,
            showgrid=False,
            zeroline=False,
            row=1, col=col,
        )
        fig.update_xaxes(showticklabels=False, row=1, col=col)
        fig.update_xaxes(
            range=[DATES[0], DATES[-1] + pd.Timedelta(days=1)],
            tickformat="%b",
            dtick="M2",
            row=2, col=col,
        )

    ymax = max(radiance_values) if radiance_values else 1
    for col in range(1, ncols + 1):
        fig.update_yaxes(
            range=[0, max(1, ymax * 1.04)],
            gridcolor="#E8EDF3",
            row=2, col=col,
        )

    fig.update_yaxes(title_text="SC", row=1, col=1)
    fig.update_yaxes(
        title_text="Radiance (nW·cm⁻²·sr⁻¹)",
        row=2, col=1,
    )

    fig.update_layout(
        template="plotly_white",
        width=width,
        height=height,
        title=dict(text=title, x=0.02, font=dict(size=17)),
        font=dict(color="#233B60", size=11),
        legend=dict(
            orientation="v",
            x=1.01, xanchor="right",
            y=0.85, yanchor="top",
            bgcolor="rgba(255,255,255,0.86)",
            font=dict(size=12),
        ),
        coloraxis=dict(
            colorscale="Greens",
            cmin=0,
            cmax=100,
            colorbar=dict(
                title=dict(text="SC (%)", side="top"),
                x=1.01,
                y=0.98,
                len=0.35,
                thickness=15,
                tickmode="array",
                tickvals=[0, 50, 100],
                ticktext=["0", "50", "100"],
                tickfont=dict(size=10),
            ),
        ),
        margin=dict(l=80, r=125, t=65, b=40),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )

    png = FIG_DIR / f"{filename}.png"
    fig.write_image(str(png), width=width, height=height, scale=2)
    display(Image(filename=str(png)))
    return png

In [ ]:
POI_FIGURES = {}

location = "Tacloban_City"
POI_FIGURES[location] = plot_source_signal(
    title="Tacloban City Center · 2013",
    panels=[
        (KERNEL_LABELS[size], poi_panel(location, size))
        for size in KERNELS
    ],
    filename="poi_tacloban_city_2013",
)

In [ ]:
for location, label in POIS.items():
    if location == "Tacloban_City":
        continue

    POI_FIGURES[location] = plot_source_signal(
        title=f"{label} · 2013",
        panels=[
            (KERNEL_LABELS[size], poi_panel(location, size))
            for size in KERNELS
        ],
        filename=f"poi_{location.lower()}_2013",
    )

In [ ]:
GHSL_PIXEL_COLORS = {
    "10": "#7ab6f5",  # Water
    "11": "#cdf57a",  # Very low density rural
    "12": "#abcd66",  # Low density rural
    "13": "#375623",  # Rural cluster
    "21": "#ffff00",  # Suburban / peri-urban
    "22": "#a87000",  # Semi-dense urban cluster
    "23": "#732600",  # Dense urban cluster
    "30": "#ff0000",  # Urban centre
}

GHSL_FIGURES = {}

for code, label in GHSL_CLASSES.items():
    GHSL_FIGURES[code] = plot_source_signal(
        title=f"GHSL {code} · {label}",
        panels=[(f"Class {code}", ghsl_panel(code))],
        filename=f"ghsl_class_{code}_2013",
        direct_color=GHSL_PIXEL_COLORS[str(code)],
        show_median_sd=True,
    )

In [ ]:
def daily_pair_metrics(signal, admitted, load):
    pair = pd.DataFrame({
        "ntl": signal.where(admitted),
        "load": load,
    }).dropna()

    n_pct = 100 * len(pair) / load.notna().sum()
    r = rmse_z = np.nan

    if (len(pair) >= 3
            and pair["ntl"].std(ddof=0) > 0
            and pair["load"].std(ddof=0) > 0):
        x = (pair["ntl"] - pair["ntl"].mean()) / pair["ntl"].std(ddof=0)
        y = (pair["load"] - pair["load"].mean()) / pair["load"].std(ddof=0)
        r = float(x.corr(y))
        rmse_z = float(np.sqrt(np.mean((x - y) ** 2)))

    return len(pair), n_pct, r, rmse_z

In [ ]:
# Shared RQ1 regional inputs for the thematic-mask figures below.
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
MASK_DIR = ROOT / "datasets/vnp46a2/ghsl_masked_ntl_samar_leyte"
NGCP_CSV = ROOT / "datasets/ngcp/NGCP_Hourly_Demand.csv"
OUT_DIR = ROOT / "outputs/figures/supplementary"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATES = pd.date_range("2013-01-01", "2024-12-31", freq="D")

GROUPS = [
    ("G8", "10-30", "All Land", "#4E79A7"),
    ("G7", "11-30", "Rural Settlements", "#A8CE74"),
    ("G6", "12-30", "Low–Mid Rural", "#7BAF54"),
    ("G5", "13-30", "Built-Up / Rural Mix", "#375623"),
    ("G4", "21-30", "Urban Fringe", "#D9B900"),
    ("G3", "22-30", "Core & Inner", "#E6A23C"),
    ("G2", "23-30", "Dense Urban", "#A87000"),
    ("G1", "30", "Urban Core", "#D65A4A"),
]

def read_ngcp_1am():
    if not NGCP_CSV.is_file():
        raise FileNotFoundError(NGCP_CSV)
    f = pd.read_csv(NGCP_CSV, skiprows=2)
    if f.shape[1] < 4:
        raise ValueError("NGCP CSV layout differs from the RQ1 hourly file")
    f.columns = (["Date", "Coincident Peak", "Hour No"]
                 + [str(i) for i in range(1, f.shape[1] - 2)])
    f["Date"] = pd.to_datetime(f["Date"], dayfirst=True, errors="coerce").dt.normalize()
    f["1"] = pd.to_numeric(f["1"], errors="coerce")
    f = f.dropna(subset=["Date"]).set_index("Date").sort_index()
    if f.index.duplicated().any():
        raise ValueError("Duplicate NGCP dates")
    return f["1"].reindex(DATES)


In [ ]:
# None displays raw radiance. Set to 95 later to cap values at P95.
CAP_PERCENTILE = 95
MARK_ABOVE_PERCENTILE = 95
SC_THRESHOLDS = [None, 25, 50, 75]

def read_direct_mask(suffix):
    path = MASK_DIR / f"EasternVisayas_DNBBRDF_{suffix}_stats.csv"
    if not path.is_file():
        raise FileNotFoundError(path)

    d = pd.read_csv(path)
    if "date" not in d:
        if "system:index" not in d:
            raise ValueError(f"{path.name}: no date or system:index")
        d["date"] = (
            d["system:index"].astype(str)
            .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
            .str.replace("_", "-", regex=False)
        )

    d["date"] = pd.to_datetime(d["date"], errors="raise").dt.normalize()
    if d["date"].duplicated().any():
        raise ValueError(f"{path.name}: duplicate dates")

    d = d.set_index("date").sort_index()
    d["Valid_px"] = pd.to_numeric(d["Valid_px"], errors="coerce")
    d["raw"] = pd.to_numeric(d["Mean_by_sum"], errors="coerce")

    # Use the full export to define each thematic mask's SC denominator.
    denominator = d["Valid_px"].max()
    if not np.isfinite(denominator) or denominator <= 0:
        raise ValueError(f"{path.name}: no valid pixels")

    d["SC"] = 100 * d["Valid_px"] / denominator
    d["raw"] = d["raw"].where(d["Valid_px"] > 0)
    return d[["raw", "SC"]].reindex(DATES)


# Load each mask once; all four figures use identical source values.
MASK_DATA = {
    code: read_direct_mask(suffix)
    for code, suffix, label, color in GROUPS
}
LOAD = read_ngcp_1am()
ORDERED_GROUPS = list(reversed(GROUPS))  # G1 through G8

def plot_daily_masks(sc_threshold):
    threshold_label = (
        "No SC threshold" if sc_threshold is None
        else f"SC ≥ {sc_threshold}%"
    )

    fig = make_subplots(
        rows=9, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.012,
        row_heights=[0.18] + [0.1025] * 8,
    )

    fig.add_trace(
        go.Scattergl(
            x=DATES, y=LOAD,
            mode="lines",
            line=dict(color="#171717", width=1.7),
            name="NGCP load (1 AM)",
            connectgaps=False,
            hovertemplate=(
                "%{x|%d %b %Y}<br>%{y:.1f} MW"
                "<extra>NGCP load</extra>"
            ),
        ),
        row=1, col=1,
    )
    fig.update_yaxes(
        title_text="MW", rangemode="tozero",
        nticks=3, gridcolor="#E8EDF3",
        row=1, col=1,
    )
    fig.add_annotation(
        x=1.01, xref="paper", y=0.5, yref="y domain",
        text="NGCP load · 1 AM",
        showarrow=False, xanchor="left",
        font=dict(size=11, color="#233B60"),
    )

    coverage = []

    for row, (code, suffix, label, color) in enumerate(
        ORDERED_GROUPS, start=2
    ):
        d = MASK_DATA[code]
        raw = d["raw"]
        sc = d["SC"]

        admitted = raw.notna()
        if sc_threshold is not None:
            admitted &= sc >= sc_threshold

        # One fixed, unfiltered reference distribution per mask.
        # Its percentile does not change between the four figures.
        mark_limit = raw.quantile(MARK_ABOVE_PERCENTILE / 100)

        if CAP_PERCENTILE is None:
            shown = raw.copy()
            high = admitted & (raw > mark_limit)
        else:
            cap_limit = raw.quantile(CAP_PERCENTILE / 100)
            shown = raw.clip(upper=cap_limit)
            high = admitted & (raw > cap_limit)

        ordinary = admitted & ~high
        custom = np.column_stack([
            sc.to_numpy(dtype=float),
            raw.to_numpy(dtype=float),
        ])

        # Small dots are the actual daily direct observations.
        fig.add_trace(
            go.Scattergl(
                x=DATES,
                y=shown.where(ordinary),
                mode="markers",
                marker=dict(size=3, color=color, opacity=0.52),
                name=f"{code}: {label}",
                customdata=custom,
                hovertemplate=(
                    "%{x|%d %b %Y}<br>"
                    "Raw %{customdata[1]:.2f} nW·cm⁻²·sr⁻¹<br>"
                    "SC %{customdata[0]:.1f}%"
                    f"<extra>{code}: {label}</extra>"
                ),
            ),
            row=row, col=1,
        )

        # Open triangles identify values above the selected percentile.
        # With CAP_PERCENTILE=None, they remain at their RAW values.
        fig.add_trace(
            go.Scatter(
                x=DATES,
                y=shown.where(high),
                mode="markers",
                marker=dict(
                    symbol="triangle-up-open",
                    size=6,
                    color=color,
                    line=dict(color="#233B60", width=0.8),
                ),
                name=f"{code} upper tail",
                customdata=custom,
                hovertemplate=(
                    "%{x|%d %b %Y}<br>"
                    "Raw %{customdata[1]:.2f} nW·cm⁻²·sr⁻¹<br>"
                    "Displayed %{y:.2f}<br>"
                    "SC %{customdata[0]:.1f}%"
                    f"<extra>{code}: {label} · upper tail</extra>"
                ),
            ),
            row=row, col=1,
        )

        n_pairs, n_pct, r, rmse_z = daily_pair_metrics(
            shown, admitted, LOAD
        )

        coverage.append({
            "mask": code,
            "SC threshold": threshold_label,
            "paired days": n_pairs,
            "N%": n_pct,
            "r": r,
            "RMSE_z": rmse_z,
        })

        fig.update_yaxes(
            rangemode="tozero", nticks=3,
            gridcolor="#E8EDF3",
            tickfont=dict(size=10),
            row=row, col=1,
        )

        r_text = "—" if not np.isfinite(r) else f"{r:.2f}"
        rmse_text = "—" if not np.isfinite(rmse_z) else f"{rmse_z:.2f}"

        fig.add_annotation(
            x=1.01, xref="paper",
            y=0.5, yref=f"y{row} domain",
            text=(
                f"{code} · {label}<br>"
                f"r = {r_text}<br>"
                f"RMSE (z) = {rmse_text}<br>"
                f"N% = {n_pct:.1f}%"
            ),
            showarrow=False,
            xanchor="left",
            align="left",
            font=dict(size=10, color="#233B60"),
        )

    for day, label in [
        ("2013-11-08", "Haiyan"),
        ("2021-12-16", "Rai"),
    ]:
        event = pd.Timestamp(day)
        for row in range(1, 10):
            fig.add_vline(
                x=event, row=row, col=1,
                line_dash="dash",
                line_color="#233B60",
                line_width=1,
            )
        fig.add_annotation(
            x=event, y=1.005, yref="paper",
            text=label, showarrow=False,
            yanchor="bottom", bgcolor="white",
            font=dict(size=11, color="#233B60"),
        )

    fig.update_xaxes(
        range=[DATES[0], DATES[-1]],
        dtick="M24", tickformat="%Y",
        gridcolor="#E8EDF3",
    )
    fig.update_xaxes(title_text="Date", row=9, col=1)

    fig.add_annotation(
        x=-0.075, xref="paper",
        y=0.41, yref="paper",
        text="Radiance (nW·cm⁻²·sr⁻¹)",
        textangle=-90, showarrow=False,
        font=dict(size=12, color="#233B60"),
    )

    cap_label = (
        "raw radiance" if CAP_PERCENTILE is None
        else f"P{CAP_PERCENTILE} display cap"
    )
    fig.update_layout(
        template="plotly_white",
        width=1250, height=1050,
        title=dict(
            text=f"Daily DNB-BRDF by GHSL mask · {threshold_label}"
                 f" · 2013–2024 · {cap_label}",
            x=0.04,
        ),
        showlegend=False,
        font=dict(color="#233B60", size=11),
        margin=dict(l=110, r=220, t=90, b=65),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )

    fig.show()

    tag = "none" if sc_threshold is None else str(sc_threshold)
    png = OUT_DIR / f"ghsl_daily_sc_{tag}_raw.png"
    try:
        fig.write_image(str(png), scale=2)
        print("Saved:", png)
    except Exception as exc:
        print(f"Figure displayed; PNG export unavailable: {exc}")

    return coverage


all_coverage = []
for threshold in SC_THRESHOLDS:
    all_coverage.extend(plot_daily_masks(threshold))

display(
    pd.DataFrame(all_coverage)
    .pivot(index="mask", columns="SC threshold", values="N%")
    .round(1)
)

## Calendar-window sensitivity: SC τ = 0%, 25%, 50%, 75%

The curves use 1–30-day **complete trailing calendar windows** for G1–G8. Every constituent day must have direct SC-qualified DNB-BRDF and NGCP 1 AM load; both series are averaged over the same window. Thus Pearson r and N% refer to the same paired endpoints, with N% divided by all 2013–2024 NGCP dates. At τ=0%, a direct observation is still required. This section uses the display cap selected above and never uses gap-filled NTL.

At high τ and long windows, few uninterrupted runs remain. Pearson r is undefined for fewer than three pairs or a constant paired series, so lines stop rather than being joined across missing estimates. Dotted r segments use 3–29 pairs and are exploratory; solid segments use at least 30. The checkpoint tables below each facet show G1, G5, and G8 at 4, 7, 14, and 30 days. A high r with low N% is not evidence that the window is reliable.


In [ ]:
WINDOW_TAUS = (0, 25, 50, 75)
WINDOW_DAYS = range(1, 31)
WINDOW_CHECKPOINTS = (4, 7, 14, 30)
WINDOW_TABLE_DIR = ROOT / "outputs/tables/supplementary/window_sensitivity"
WINDOW_TABLE_DIR.mkdir(parents=True, exist_ok=True)

n_possible = int(LOAD.notna().sum())
if n_possible == 0:
    raise ValueError("No 2013–2024 NGCP 1 AM dates were loaded")

window_rows = []
for code, suffix, label, color in reversed(GROUPS):
    d = MASK_DATA[code]
    raw = d["raw"]
    if CAP_PERCENTILE is None:
        radiance = raw
    else:
        radiance = raw.clip(upper=raw.quantile(CAP_PERCENTILE / 100))

    for tau in WINDOW_TAUS:
        admitted = raw.notna() & (d["SC"] >= tau)
        direct = radiance.where(admitted)
        for width in WINDOW_DAYS:
            # min_periods=width: every day in this trailing calendar window
            # must have a direct SC-qualified NTL value and an NGCP value.
            ntl_window = direct.rolling(width, min_periods=width).mean()
            load_window = LOAD.rolling(width, min_periods=width).mean()
            paired = pd.concat(
                [ntl_window.rename("ntl"), load_window.rename("load")], axis=1
            ).dropna()
            n = len(paired)
            r = np.nan
            if n < 3:
                r_status = "fewer than 3 paired dates"
            elif (paired["ntl"].std(ddof=0) == 0
                  or paired["load"].std(ddof=0) == 0):
                r_status = "constant paired series"
            else:
                r = float(paired["ntl"].corr(paired["load"]))
                r_status = "defined"
            window_rows.append({
                "mask": code, "label": label, "tau": tau,
                "window_days": width, "r": r, "r_status": r_status,
                "n_pairs": n, "n_possible": n_possible,
                "n_pct": 100 * n / n_possible,
            })

WINDOW_METRICS = pd.DataFrame(window_rows)
for _, group in WINDOW_METRICS.groupby(["mask", "tau"]):
    counts = group.sort_values("window_days")["n_pairs"]
    assert counts.diff().dropna().le(0).all(), "Complete-window N must not rise with width"

WINDOW_METRICS.to_csv(
    WINDOW_TABLE_DIR / "ghsl_complete_window_tau0_25_50_75_2013_2024.csv", index=False
)
WINDOW_CHECKPOINT_TABLE = WINDOW_METRICS[
    WINDOW_METRICS.window_days.isin(WINDOW_CHECKPOINTS)
].copy()
WINDOW_CHECKPOINT_TABLE.to_csv(
    WINDOW_TABLE_DIR / "ghsl_window_checkpoints_tau0_25_50_75.csv", index=False
)
WINDOW_BREAKS = (WINDOW_METRICS[WINDOW_METRICS.r.isna()]
                 .sort_values("window_days")
                 .groupby(["tau", "mask"], as_index=False).first()
                 [["tau", "mask", "window_days", "n_pairs", "r_status"]])
WINDOW_BREAKS.to_csv(
    WINDOW_TABLE_DIR / "ghsl_first_undefined_r_by_mask_tau.csv", index=False
)
print(f"Saved {len(WINDOW_METRICS)} mask × SC × window results; NGCP denominator = {n_possible} days")
display(WINDOW_CHECKPOINT_TABLE[
    WINDOW_CHECKPOINT_TABLE["mask"].isin(["G1", "G5", "G8"])
][["mask", "tau", "window_days", "r", "r_status", "n_pairs", "n_pct"]].round(3))
display(WINDOW_BREAKS)


In [ ]:
WINDOW_COLOR = {code: color for code, suffix, label, color in GROUPS}
WINDOW_MIN_SOLID_PAIRS = 30  # visual support cue; all computed r and n remain in CSV

def window_checkpoint_text(tau, code, width, metric):
    q = WINDOW_METRICS[
        (WINDOW_METRICS.tau == tau) & (WINDOW_METRICS["mask"] == code)
        & (WINDOW_METRICS.window_days == width)
    ].iloc[0]
    if metric == "n_pct":
        return f"{q.n_pct:.1f}%"
    if not np.isfinite(q.r):
        return "—"
    return f"{q.r:.2f}" + ("*" if q.n_pairs < WINDOW_MIN_SOLID_PAIRS else "")

def plot_window_metric(metric, title, y_title, filename):
    fig = make_subplots(
        rows=2, cols=4, shared_yaxes=True,
        row_heights=[0.78, 0.22], vertical_spacing=0.08,
        horizontal_spacing=0.045,
        specs=[[{"type": "xy"}] * 4, [{"type": "table"}] * 4],
        subplot_titles=[f"SC τ={tau}%" for tau in WINDOW_TAUS] + [""] * 4,
    )
    table_values = {}
    for col, tau in enumerate(WINDOW_TAUS, start=1):
        if metric == "r":
            valid = WINDOW_METRICS[
                (WINDOW_METRICS.tau == tau) & WINDOW_METRICS.r.notna()
            ]
            if len(valid):
                last = int(valid.window_days.max())
                if last < 30:
                    fig.add_vrect(
                        x0=last + 0.5, x1=30, row=1, col=col,
                        fillcolor="#EFF2F6", opacity=0.75,
                        line_width=0, layer="below",
                    )

        for code, suffix, label, color in reversed(GROUPS):
            q = WINDOW_METRICS[
                (WINDOW_METRICS.tau == tau) & (WINDOW_METRICS["mask"] == code)
            ].sort_values("window_days")
            custom = np.column_stack([q.n_pairs, q.n_pct])
            hover = (
                "Window %{x} d<br>Value %{y:.3f}<br>Paired days %{customdata[0]:.0f}"
                "<br>N %{customdata[1]:.1f}%<extra>%{fullData.name}</extra>"
            )
            if metric == "r":
                solid = q.r.where(q.n_pairs >= WINDOW_MIN_SOLID_PAIRS)
                sparse = q.r.where((q.n_pairs >= 3) &
                                   (q.n_pairs < WINDOW_MIN_SOLID_PAIRS))
                fig.add_trace(go.Scatter(
                    x=q.window_days, y=solid, mode="lines",
                    name=f"{code} {label}", showlegend=(col == 1),
                    line=dict(color=color, width=2), connectgaps=False,
                    customdata=custom, hovertemplate=hover,
                ), row=1, col=col)
                fig.add_trace(go.Scatter(
                    x=q.window_days, y=sparse, mode="lines",
                    name=f"{code} {label} · <30 pairs", showlegend=False,
                    line=dict(color=color, width=1.7, dash="dot"),
                    opacity=0.65, connectgaps=False,
                    customdata=custom, hovertemplate=hover,
                ), row=1, col=col)
            else:
                fig.add_trace(go.Scatter(
                    x=q.window_days, y=q.n_pct, mode="lines",
                    name=f"{code} {label}", showlegend=(col == 1),
                    line=dict(color=color, width=2), connectgaps=False,
                    customdata=custom, hovertemplate=hover,
                ), row=1, col=col)
            marks = q[q.window_days.isin(WINDOW_CHECKPOINTS)]
            fig.add_trace(go.Scatter(
                x=marks.window_days, y=marks[metric], mode="markers",
                showlegend=False, hoverinfo="skip",
                marker=dict(color=color, size=4.5),
            ), row=1, col=col)

        for width in WINDOW_CHECKPOINTS:
            fig.add_vline(x=width, row=1, col=col, line_dash="dot",
                          line_color="#DCE4ED", line_width=1)
        fig.update_xaxes(
            title_text="Calendar window (days)",
            tickvals=[1, 4, 7, 14, 21, 30], range=[1, 30],
            row=1, col=col,
        )

        codes = ["G1", "G5", "G8"]
        values = [codes] + [
            [window_checkpoint_text(tau, code, width, metric) for code in codes]
            for width in WINDOW_CHECKPOINTS
        ]
        table_values[col] = values

    # Plotly's add_vline/add_vrect inspect every trace for xaxis/yaxis.
    # go.Table has neither, so add all XY shapes above before any tables.
    for col, values in table_values.items():
        fig.add_trace(go.Table(
            header=dict(
                values=["Mask", "4 d", "7 d", "14 d", "30 d"],
                fill_color="#E9EFF6", align="center",
                font=dict(color="#233B60", size=10), height=21,
                line_color="#DCE4ED",
            ),
            cells=dict(
                values=values, fill_color="white", align="center",
                font=dict(color="#233B60", size=10), height=21,
                line_color="#E8EDF3",
            ),
            columnwidth=[1.1, 1, 1, 1, 1],
        ), row=2, col=col)

    fig.update_yaxes(
        title_text=y_title, row=1, col=1,
        range=[-1, 1.03] if metric == "r" else [0, 100],
    )
    fig.update_layout(
        template="plotly_white", width=1800, height=650,
        title=title + " · complete direct windows · 2013–2024",
        legend=dict(x=1.01, y=1, xanchor="left", yanchor="top",
                    font=dict(size=10)),
        margin=dict(l=75, r=215, t=90, b=55),
        font=dict(color="#233B60", size=11),
        paper_bgcolor="white", plot_bgcolor="white",
    )
    fig.update_xaxes(gridcolor="#E8EDF3")
    fig.update_yaxes(gridcolor="#E8EDF3")
    if metric == "r":
        fig.add_annotation(
            x=0.5, y=-0.04, xref="paper", yref="paper",
            text="Dotted: 3–29 paired dates (*) · blank: r undefined (<3 pairs or constant series)",
            showarrow=False, font=dict(size=10, color="#62758E"),
        )
    fig.show()
    fig.write_html(OUT_DIR / f"{filename}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(str(OUT_DIR / f"{filename}.png"), scale=2)
    except Exception as exc:
        print(f"{filename}: HTML saved; PNG unavailable ({exc})")
    return fig

WINDOW_R_FIG = plot_window_metric(
    "r", "Alignment across calendar windows and SC thresholds",
    "Pearson r", "ghsl_r_window_tau0_25_50_75",
)
WINDOW_N_FIG = plot_window_metric(
    "n_pct", "Retained coverage across calendar windows and SC thresholds",
    "N% retained coverage", "ghsl_n_window_tau0_25_50_75",
)
